<a href="https://colab.research.google.com/github/martim-p05/detecao-anatomia-abelhas/blob/bee_species_identification/01_preprocessing_20_species.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from google.colab import drive
import os
from pathlib import Path
import numpy as np
from PIL import Image

# 1. Mount Google Drive
drive.mount('/content/drive')
print("Google Drive mounted successfully.\n")

# 2. Define Constants and Paths
IMG_SIZE = (224, 224)
working_folder = Path('/content/drive/MyDrive/robin_2/')
data_dir = working_folder / 'data'
temp_dir = working_folder / 'temp_files'
output_filepath = temp_dir / 'dataset_20_species.npz'

temp_dir.mkdir(parents=True, exist_ok=True)

# 3. Find Species Folders
if data_dir.exists():
    list_species = [d.name for d in data_dir.iterdir() if d.is_dir()]
else:
    raise FileNotFoundError(f"Directory not found: {data_dir}")

# 4. Scan for Images
image_paths = []
labels = []
image_extensions = ('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tif')

print("Scanning for images...")
for species_name in list_species:
    species_dir = data_dir / species_name
    for file_path in species_dir.rglob('*'):
        if file_path.is_file() and file_path.suffix.lower() in image_extensions:
            image_paths.append(file_path)
            labels.append(species_name)

num_images = len(image_paths)
print(f"TOTAL: Found {num_images} images.\n")

# 5. Process and Resize Images (MEMORY OPTIMIZED)
print("Starting image processing... (Memory optimized mode)")

# PRE-ALLOCATE MEMORY to prevent RAM crash (using float32 cuts RAM usage by 50%)
X = np.zeros((num_images, IMG_SIZE[0], IMG_SIZE[1], 3), dtype=np.float32)

for count, img_path in enumerate(image_paths):
    try:
        img = Image.open(img_path).convert('RGB')
        img = img.resize(IMG_SIZE)
        # Store directly into the pre-allocated array
        X[count] = np.array(img, dtype=np.float32) / 255.0

        if (count + 1) % 500 == 0 or (count + 1) == num_images:
            print(f"  Processed {count + 1}/{num_images} images...")
    except Exception as e:
        print(f"  Error processing {img_path}: {e}")

print(f"\nShape of final image array (X): {X.shape}")

# 6. Save to compressed .npz file
print(f"Compressing and saving data to Drive. This might take a few minutes...")
np.savez_compressed(output_filepath, X=X, labels=labels)

print(f"\nSUCCESS! Dataset saved successfully to: {output_filepath}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully.

Scanning for images...
TOTAL: Found 6834 images.

Starting image processing... (Memory optimized mode)
  Processed 500/6834 images...
  Processed 1000/6834 images...
  Processed 1500/6834 images...
  Processed 2000/6834 images...
  Processed 2500/6834 images...
  Processed 3000/6834 images...
  Processed 3500/6834 images...
  Processed 4000/6834 images...
  Processed 4500/6834 images...
  Processed 5000/6834 images...
  Processed 5500/6834 images...
  Processed 6000/6834 images...
  Processed 6500/6834 images...
  Processed 6834/6834 images...

Shape of final image array (X): (6834, 224, 224, 3)
Compressing and saving data to Drive. This might take a few minutes...

SUCCESS! Dataset saved successfully to: /content/drive/MyDrive/robin_2/temp_files/dataset_20_species.npz
